In [1]:
import pandas as pd 
import numpy as np 
import random 
import os 
import json 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime, timedelta 
import funcoes

In [2]:
df = pd.read_csv('../dados/raw/vendas.csv')
df_original = df.copy()

In [3]:
# Visão geral do df
df_original

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0
...,...,...,...,...,...,...,...,...
145,146,2024-08-09,Cliente_019,Notebook,Computadores,Norte,2.0,3500.0
146,147,2024-12-09,Cliente_028,Teclado,Periféricos,Sudeste,6.0,250.0
147,148,2024-06-08,Cliente_008,Tablet,Celulares,Sudeste,10.0,1800.0
148,149,2024-07-07,Cliente_018,Tablet,Celulares,Norte,9.0,1800.0


In [4]:
# Informações relevantes para investigar os dados
funcoes.analise_dados(df_original)

,Tipos de Dados,Qtd_Não_Nulos,Valores Únicos,Qtd_Nulos,% Nulos,Duplicados
id_venda,int64,150,150,0,0.0%,0
data_venda,object,150,113,0,0.0%,0
cliente,object,150,30,0,0.0%,0
produto,object,150,9,0,0.0%,0
categoria,object,150,3,0,0.0%,0
regiao,object,150,5,0,0.0%,0
quantidade,float64,144,10,6,4.0%,0
preco_unitario,float64,147,6,3,2.0%,0


In [5]:
# Inspecionando elementos e checagem geral
funcoes.contagem_valores(df_original)

--- RELATÓRIO DE VALORES ÚNICOS E CONTAGENS ---

[ ID_VENDA ] -> 150 valores. (Omitido)

[ DATA_VENDA ]: 113 Elementos unicos
'2024-05-15' (10 caracteres): 5  |  'DATA INVALIDA' (13 caracteres): 4  |  '2024-11-22' (10 caracteres): 4
'2024-02-08' (10 caracteres): 3  |  '2024-10-26' (10 caracteres): 3  |  '2024-11-01' (10 caracteres): 2
'2024-07-21' (10 caracteres): 2  |  '2024-05-30' (10 caracteres): 2  |  '2024-09-16' (10 caracteres): 2
'2024-07-29' (10 caracteres): 2  |  '2024-05-05' (10 caracteres): 2  |  '2024-05-08' (10 caracteres): 2
'2024-02-25' (10 caracteres): 2  |  '2024-07-23' (10 caracteres): 2  |  '2024-10-22' (10 caracteres): 2
'2024-01-17' (10 caracteres): 2  |  '2024-11-05' (10 caracteres): 2  |  '2024-08-28' (10 caracteres): 2
'2024-08-08' (10 caracteres): 2  |  '2024-04-22' (10 caracteres): 2  |  '2024-12-03' (10 caracteres): 2
'2024-03-07' (10 caracteres): 2  |  '2024-03-20' (10 caracteres): 2  |  '2024-06-16' (10 caracteres): 2
'2024-07-04' (10 caracteres): 2  |  '20

In [6]:
'''ANALISE INICIAL: Corrigir 4 colunas 

'data_venda' ---> Tem um valor 'DATA INVALIDA' e não está no formato datetime
'quantidade' e 'preco_unitario' ---> Poucos valores Nulos 
'produto' ---> Erro de formatação, espaços vazios '''

"ANALISE INICIAL: Corrigir 4 colunas \n\n'data_venda' ---> Tem um valor 'DATA INVALIDA' e não está no formato datetime\n'quantidade' e 'preco_unitario' ---> Poucos valores Nulos \n'produto' ---> Erro de formatação, espaços vazios "

In [7]:
# Retirando espaço vazio da coluna 'produtos' e conferindo
display(df_original['produto'].value_counts().items)
df_formatado = df_original.copy()

df_formatado['produto'] = df_formatado['produto'].str.strip()
df_formatado['produto'].value_counts().items

<bound method Series.items of produto
Notebook        27
Smartphone      25
Tablet          25
Mouse           24
Monitor         24
Teclado         22
Tablet           1
Smartphone       1
 Smartphone      1
Name: count, dtype: int64>

<bound method Series.items of produto
Notebook      27
Smartphone    27
Tablet        26
Mouse         24
Monitor       24
Teclado       22
Name: count, dtype: int64>

In [8]:
# Verificando contexto dos nulos
todas_linhas_com_nulo = df_formatado[df_formatado.isna().any(axis=1)]
todas_linhas_com_nulo

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
21,22,DATA INVALIDA,Cliente_023,Smartphone,Celulares,Sudeste,NaN,2200.0
48,49,2024-05-15,Cliente_014,Smartphone,Celulares,Norte,NaN,2200.0
63,64,2024-10-26,Cliente_016,Monitor,Computadores,Sudeste,7.0,NaN
78,79,2024-06-21,Cliente_026,Teclado,Periféricos,Nordeste,NaN,250.0
84,85,2024-07-03,Cliente_002,Tablet,Celulares,Norte,4.0,NaN
95,96,2024-08-12,Cliente_009,Teclado,Periféricos,Sudeste,8.0,NaN
113,114,2024-10-04,Cliente_008,Tablet,Celulares,Sudeste,NaN,1800.0
126,127,2024-07-21,Cliente_022,Tablet,Celulares,Nordeste,NaN,1800.0


In [9]:
# Conferindo se é possivel conectar os valores
df_verdade = df_formatado.groupby('produto').agg(lambda x: pd.Series(x.unique()).sort_values(na_position='last').tolist())
df_verdade

,id_venda,data_venda,cliente,categoria,regiao,quantidade,preco_unitario
produto,,,,,,,
Monitor,"[13, 14, 26, 46, 53, 57, 59, 64, 76, 87, 88, 9...","[2024-01-25, 2024-01-28, 2024-02-24, 2024-02-2...","[Cliente_001, Cliente_003, Cliente_005, Client...",[Computadores],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1200.0, nan]"
Mouse,"[1, 3, 4, 15, 21, 30, 38, 44, 45, 48, 51, 52, ...","[2024-01-13, 2024-02-08, 2024-02-12, 2024-02-1...","[Cliente_001, Cliente_002, Cliente_004, Client...",[Periféricos],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]",[120.0]
Notebook,"[2, 6, 7, 10, 18, 19, 28, 29, 31, 35, 37, 40, ...","[2024-01-10, 2024-01-17, 2024-01-22, 2024-01-2...","[Cliente_003, Cliente_004, Cliente_006, Client...",[Computadores],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",[3500.0]
Smartphone,"[8, 11, 17, 22, 23, 24, 27, 32, 34, 47, 49, 66...","[2024-01-17, 2024-02-10, 2024-02-25, 2024-04-2...","[Cliente_001, Cliente_004, Cliente_007, Client...",[Celulares],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",[2200.0]
Tablet,"[5, 25, 36, 43, 54, 61, 65, 78, 80, 82, 83, 85...","[2024-02-29, 2024-03-07, 2024-04-17, 2024-04-2...","[Cliente_002, Cliente_003, Cliente_004, Client...",[Celulares],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1800.0, nan]"
Teclado,"[9, 12, 16, 20, 33, 39, 42, 50, 55, 60, 79, 94...","[2024-01-08, 2024-01-14, 2024-01-21, 2024-03-2...","[Cliente_003, Cliente_005, Cliente_008, Client...",[Periféricos],"[Centro-Oeste, Nordeste, Norte, Sudeste, Sul]","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[250.0, nan]"


In [10]:
# Substitui os NaNs da coluna 'preco' pelo outro valor
df_formatado['preco_unitario'] = df_formatado['preco_unitario'].fillna(df_formatado.groupby('produto')['preco_unitario'].transform('max'))

In [11]:
# Verificando contexto dos nulos
todas_linhas_com_nulo = df_formatado[df_formatado.isna().any(axis=1)]
todas_linhas_com_nulo


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
21,22,DATA INVALIDA,Cliente_023,Smartphone,Celulares,Sudeste,NaN,2200.0
48,49,2024-05-15,Cliente_014,Smartphone,Celulares,Norte,NaN,2200.0
78,79,2024-06-21,Cliente_026,Teclado,Periféricos,Nordeste,NaN,250.0
113,114,2024-10-04,Cliente_008,Tablet,Celulares,Sudeste,NaN,1800.0
126,127,2024-07-21,Cliente_022,Tablet,Celulares,Nordeste,NaN,1800.0


In [12]:
# Verificando distribuição para decidir o que fazer com os NaN da coluna 'quantidade'
assimetria = df_formatado['quantidade'].skew()
print(f"Assimetria: {assimetria}")

df_estatisticas = df_formatado.groupby('produto')['quantidade'].agg(['mean', 'median']).reset_index()

df_estatisticas.round(2)

Assimetria: 0.059368793750001765


,produto,mean,median
0,Monitor,5.42,6.5
1,Mouse,4.96,4.5
2,Notebook,5.15,4.5
3,Smartphone,5.60,6.0
4,Tablet,5.38,5.0
5,Teclado,6.10,6.0


In [13]:
# Preenche os nulos com a mediana específica de cada produto
df_formatado['quantidade'] = df_formatado['quantidade'].fillna(df_formatado.groupby('produto')['quantidade'].transform('median'))

In [14]:
# Conferindo alterações
indices_alterados = df_original.compare(df_formatado).index

# 2. Filtra o seu dataset final para mostrar apenas essas linhas inteiras
linhas_modificadas = df_formatado.loc[indices_alterados]

linhas_modificadas


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,4.5,3500.0
21,22,DATA INVALIDA,Cliente_023,Smartphone,Celulares,Sudeste,6.0,2200.0
48,49,2024-05-15,Cliente_014,Smartphone,Celulares,Norte,6.0,2200.0
63,64,2024-10-26,Cliente_016,Monitor,Computadores,Sudeste,7.0,1200.0
78,79,2024-06-21,Cliente_026,Teclado,Periféricos,Nordeste,6.0,250.0
79,80,2024-11-01,Cliente_016,Tablet,Celulares,Norte,5.0,1800.0
80,81,2024-11-26,Cliente_022,Smartphone,Celulares,Norte,8.0,2200.0
84,85,2024-07-03,Cliente_002,Tablet,Celulares,Norte,4.0,1800.0
95,96,2024-08-12,Cliente_009,Teclado,Periféricos,Sudeste,8.0,250.0
113,114,2024-10-04,Cliente_008,Tablet,Celulares,Sudeste,5.0,1800.0


In [15]:
# Convertendo para datetime
df_formatado['data_venda'] = pd.to_datetime(df_formatado['data_venda'], errors='coerce')
print(df_formatado['data_venda'].dtype)

datetime64[ns]


In [16]:
# Conferindo
funcoes.analise_dados(df_formatado)

,Tipos de Dados,Qtd_Não_Nulos,Valores Únicos,Qtd_Nulos,% Nulos,Duplicados
id_venda,int64,150,150,0,0.0%,0
data_venda,datetime64[ns],146,112,4,2.67%,0
cliente,object,150,30,0,0.0%,0
produto,object,150,6,0,0.0%,0
categoria,object,150,3,0,0.0%,0
regiao,object,150,5,0,0.0%,0
quantidade,float64,150,11,0,0.0%,0
preco_unitario,float64,150,6,0,0.0%,0


In [17]:
# Decisões: Classificação de cliente(quantidade x preco_unitario), Bronze até 9000, Prata até 17600, < Ouro
funcoes.resumo_estatistico(df_formatado)

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.0,146,150,150,150,150,150.0,150.0
unique,-,-,30,6,3,5,-,-
top,-,-,Cliente_024,Notebook,Celulares,Sudeste,-,-
freq,-,-,8,27,53,42,-,-
mean,75.5,2024-07-05 22:21:22.191780864,-,-,-,-,5.42,1585.87
min,1.0,2024-01-08 00:00:00,-,-,-,-,1.0,120.0
25%,38.25,2024-04-22 00:00:00,-,-,-,-,3.0,250.0
50%,75.5,2024-07-03 12:00:00,-,-,-,-,5.0,1800.0
75%,112.75,2024-10-11 12:00:00,-,-,-,-,8.0,2200.0
max,150.0,2024-12-27 00:00:00,-,-,-,-,10.0,3500.0


In [18]:
# Adicionando coluna total
df_formatado['valor_total'] = df_formatado['preco_unitario'] * df_formatado['quantidade']

In [26]:
os.makedirs('dados/processed', exist_ok=True) 
df_formatado.to_csv("dados/processed/vendas_tratado.csv", index=False)

df_formatado.to_json(
    "dados/processed/vendas_tratado.json", 
    orient="records", 
    indent=4, 
    force_ascii=False  # Mantém os acentos e caracteres especiais em português
)